In [ ]:
# syGlass modules
import syglass as sy
import pprint

# Importing tracks from our analysis pipeline
from os import path
import pandas as pd
from IPython.display import display
import numpy as np 
import sys 
import time 
import zarr
import glob
import os
import re
import matplotlib.pyplot as plt
import dask.array as da

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)
from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

In [ ]:
# # Import syglass project
# project = sy.get_project('D:\syGlass_trial\Abhi_LLSM/controlOS_ch3_syg/controlOS_ch3_syg.syg')

# # Import the multitracking points from the project (if this hasn't been done before, it should be an empty dictionary)
# points = project.get_multitracking_points()

In [ ]:
# ## Make sure to check existing points before creating new ones
# ## This section will focus on adding points the the syGlass project
# # This assumes that your notebook is inside 'Jupyter Notebooks', which is at the same level as 'movie_data'
# base_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'movie_data')

# zarr_directory = 'zarr_file/all_channels_data'
# zarr_full_path = os.path.join(base_dir, zarr_directory)

# input_directory = os.path.join(base_dir,'datasets')

# # in the datasets directory, list pkl files that start with 'all_detections_channel'
# files = glob.glob(f'{input_directory}/track_df_c*_cleaned.pkl')

# # take the first file
# file = files[0]

# input_directory_full = os.path.join(input_directory, file)

# # find out which channel was detected
# match = re.search(r'track_df_c(\d+)_cleaned.pkl', file)
# channel_detected = match.group(1)

# track_df = pd.read_pickle(input_directory_full)

In [ ]:
base_dir = r'Z:\Abhi\LLSM_Analysis'
input_file_directory = '41OS_0-5min_analysis/'

input_directory = os.path.join(base_dir, input_file_directory) + 'datasets/track_df_cleaned_final_full.pkl'
# input_directory_filtered = os.path.join(base_dir, input_file_directory) + 'datasets/filtered_tracks_final.pkl'

zarr_file_directory = '41OS_0-5min_analysis/zarr_file/all_channels_data'

zarr_full_path = os.path.join(base_dir, zarr_file_directory)

# open tracks and movie
track_df = pd.read_pickle(input_directory)
# track_df_filtered = pd.read_pickle(input_directory_filtered)
z2 = zarr.open(zarr_full_path, mode='r')
# reset index
# track_df = track_df.reset_index(drop=True)

In [ ]:
path_to_detections = os.path.join(base_dir, input_file_directory) + 'detection/channel_1_detections.pkl'
df = pd.read_pickle(path_to_detections)

In [ ]:
# Report number of frames and a random color per track

import random

# Calculate the number of unique frames for each track and create a mapping from track_id to number_of_frames
frame_counts = track_df.groupby('track_id')['frame'].nunique().to_dict()

# Map the frame counts to the original dataframe, creating a new column
track_df['number_of_frames'] = track_df['track_id'].map(frame_counts)

# Assuming 'track_df' is your DataFrame
track_ids = track_df['track_id'].unique().tolist()

# Generate a list of random colors
colors = [random.randint(0, 255) for _ in range(len(track_ids))]

# Create a dictionary that maps each track id to a random color
color_dict = dict(zip(track_ids, colors))

# Now you can use this dictionary to assign colors to the tracks
track_df['color'] = track_df['track_id'].map(color_dict)

In [ ]:
# df = track_df[track_df['track_id'] == 33789]
# df[['frame', 'c1_voxel_sum_adjusted']]

In [ ]:
#syGlass instructions for adding points
# # add two new points to the dict. One orange point and one violet point
# pts['Orange'].append([np.array([30.02, 23.02, 19.02]), 235, 3]) # [[z,y,x], frame, series number]
# pts['Violet'].append([np.array([45.02, 22.042, 5.03]), 600, 4]) # [[z,y,x], frame, series number]

# # set the projects new and updated multi tracking points
# project.set_multitracking_points(pts)

# # retrieve the updated points
# out = project.get_multitracking_points()

# # display the dict with pretty print for organization
# pp = pprint.PrettyPrinter(indent=4)
# pp.pprint(out)

In [ ]:
# Alternative usage if you already have the DataFrame in memory

def convert_tracking_data_from_df(track_df):
    # Create a new DataFrame with the desired columns
    new_df = pd.DataFrame()
    
    # Map the columns
    new_df['SERIES'] = track_df['track_id']
    new_df['FRAME'] = track_df['frame']
    new_df['X'] = track_df['mu_x']
    new_df['Y'] = track_df['mu_y']
    new_df['Z'] = track_df['mu_z']
    
    # Determine color based on number_of_frames
    def get_color(num_frames):
        if num_frames < 20:
            return 'Red'
        elif 20 <= num_frames < 40:
            return 'Orange'
        elif 40 <= num_frames < 60:
            return 'Yellow'
        elif 60 <= num_frames < 80:
            return 'Green'
        else:  # num_frames >= 80
            return 'Cyan'
    
    new_df['COLOR'] = track_df['number_of_frames'].apply(get_color)
    
    # Reorder columns to match the desired format
    new_df = new_df[['SERIES', 'COLOR', 'FRAME', 'X', 'Y', 'Z']]
    
    return new_df

# If you already have track_df in memory:
# result_df = convert_tracking_data_from_df(track_df)
# 

In [ ]:
# new_df = convert_tracking_data_from_df(track_df)
# new_df_0 = new_df[(new_df['SERIES'] == 14291)]

In [ ]:
# #Export the DataFrame to a CSV file
# # new_df.to_csv("D:\syGlass_trial\Abhi_LLSM\TrackingExport.csv", index=False)
# new_df_0.to_csv("D:\syGlass_trial\Abhi_LLSM\TrackingExport_0.csv", index=False)

In [ ]:
# new_df_0

Suite of Visualziation and Benchmarking Tools to Assess Track Quality

In [ ]:
# # Visualizing the detections in Napari

# # Make a mask of the first time point of the detections

# import napari

# df_reset = df.reset_index()

# masks = visualize_3D_gaussians(zarr_obj = z2, gaussians_df = df_reset[df_reset['frame'] == 86])
# # masks = visualize_3D_gaussians(zarr_obj = z2, gaussians_df = df)

# # Create a napari viewer
# viewer = napari.Viewer()

# #open the zarr file in read mode
# dask_array = da.from_zarr(z2)

# # first time point of the zarr file and the channel to detect
# #the axis arrangement is (t,c,z,y,x)

# dask_array_slice = dask_array[86,2,:,:,:]

# # Add the 3D stack to the viewer
# layer_raw = viewer.add_image(dask_array_slice, name='fluorescence', interpolation3d = 'nearest', blending = 'additive', colormap = 'magenta')

# # layer_mask = viewer.add_image(masks, name = 'detections mask')
# layer_mask = viewer.add_image(masks, name = 'detections', interpolation3d = 'nearest', blending = 'additive', colormap = 'green')

# #other useful parameters 
# #color_map = list
# #contrast_limits = list of list 

# # Add Bounding Box
# layer_raw.bounding_box.visible = True

In [ ]:
track_df[track_df['track_id'] == 572][['frame', 'mu_x', 'mu_y', 'mu_z', 'c2_voxel_sum_adjusted', 'c1_vol_sig', 'c1_vol_bg']]
# track_df[track_df['track_id'] == 14942][['frame', 'mu_x', 'mu_y', 'mu_z', 'c3_voxel_sum_adjusted']]

In [ ]:
x= 175
y= 970
z = 57
frame = 18
vol_sig = 175
vol_bg = 441
channel = 1

a = z2[frame, channel, (z-3):(z+4), (y-2):(y+3), (x-2):(x+3)].sum()
b = z2[frame, channel, (z-4):(z+5), (y-3):(y+4), (x-3):(x+4)].sum()
a - (((b-a)/(vol_bg-vol_sig)) * vol_sig)

In [ ]:
# new_df_0